### Auto Encoder

In [ ]:
import sys,os
sys.path.append('../../package/')
import os
import matplotlib

# 1. make it run without a display (headless)
os.environ["MUJOCO_GL"] = "egl"

# 2. keep Matplotlib from opening windows
matplotlib.use('Agg')
%run ../../package/init_jupyter.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mujoco as mj
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import clear_output, display
from scipy.spatial import distance, procrustes
import json
sys.path.append('../')
sys.path.append('../../asset/')
sys.path.append('../../package/') # for 'preretarget'
sys.path.append('../../package/yoon/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/motion_retarget')
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/utility/')
sys.path.append('../../package/smpl/')
sys.path.append('../../package/irvae_tools/')
sys.path.append('../../package/bvh_helper/')
from motion_retarget import *
from geometry import relaxed_distortion_measure
from clustering_robots import *
from _robot_info import *
from mujoco_parser import *
from utils import *
from smpl_model import *
# from smpl_parser import *
from bvh_parser import *

plt.rc('xtick', labelsize=6)
plt.rc('ytick', labelsize=6)
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
print ("PyTorch:[%s]"%(th.__version__))

In [ ]:
device = "cuda:0"

# Train Autoencoder

In [ ]:
train=True

In [ ]:
class AutoEncoderClass(nn.Module):
    def __init__(
        self,
        x_dim,
        z_dim=2,
        h_dims=[128, 128],
        loss_type='l2',           # 'l2', 'l1', 'huber'
        actv=nn.ReLU(),           # activation function
        iso = False,
        iso_reg = 1e-7,
    ):
        super(AutoEncoderClass, self).__init__()
        self.name = 'autoencoder'
        self.z_dim = z_dim
        self.h_dims = h_dims
        self.loss_type = loss_type.lower()
        self.actv = actv
        self.iso = iso
        self.iso_reg = iso_reg

        if isinstance(x_dim, int):
            self.input_shape = (x_dim,)
            self.flattened_dim = x_dim
        elif isinstance(x_dim, tuple):
            self.input_shape = x_dim
            self.flattened_dim = int(np.prod(x_dim))
        else:
            raise ValueError("x_dim must be int or tuple")

        enc_layers = []
        in_dim = self.flattened_dim
        for h_dim in self.h_dims:
            enc_layers.append(nn.Linear(in_dim, h_dim))
            enc_layers.append(self.actv)
            in_dim = h_dim
        enc_layers.append(nn.Linear(in_dim, self.z_dim))
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        in_dim = self.z_dim
        for h_dim in reversed(self.h_dims):
            dec_layers.append(nn.Linear(in_dim, h_dim))
            dec_layers.append(self.actv)
            in_dim = h_dim
        dec_layers.append(nn.Linear(in_dim, self.flattened_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x): return self.encoder(x.view(x.size(0), -1))

    def decode(self, z): return self.decoder(z).view(z.size(0), *self.input_shape)

    def forward(self, x): z = self.encode(x); return self.decode(z), z
    
    def compute_loss(self, x, x_hat):
        if self.loss_type == 'l2': 
            loss = nn.functional.mse_loss(x_hat, x)
        elif self.loss_type == 'l1': 
            loss = nn.functional.l1_loss(x_hat, x)
        elif self.loss_type == 'huber': 
            loss = nn.functional.huber_loss(x_hat, x)
        elif self.loss_type == 'mseloss':
            loss = nn.functional.mse_loss(x_hat,x)
        else: 
            raise ValueError(f"Unknown loss_type: {self.loss_type}")
        if self.iso:
            iso_loss = relaxed_distortion_measure(self.decode, self.encode(x), eta=0.2, metric = 'identity')
            total_loss = loss + self.iso_reg * iso_loss
        else:
            iso_loss = th.tensor(0.0, device=x.device, dtype=loss.dtype)
            total_loss =loss
        return total_loss, loss, iso_loss

# ---------- data loading ----------

def load_data_subset(folder_path):
    arr = np.load(os.path.join(folder_path, 'data.npy'))  # assumes 'data.npy' exists
    print(f"array shape: {arr.shape}")
    if arr.shape[-1]==6:
        data = arr[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], :]
    elif arr.shape[-1]==5:
        data = arr[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 21,22,23,24,25,26,27,28,29,30,31], :]
    return data.reshape((30, -1))

def load_all_npy_from_folder(folder_path):
    """
    Load all .npy files in the given folder into a list

    :param folder_path: path to the folder containing .npy files
    :return: list of numpy arrays
    """
    npy_list = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.npy'):
            full_path = os.path.join(folder_path, fname)
            arr = np.load(full_path)
            npy_list.append(arr)
    return npy_list

def load_npy_as_dict(folder_path, remove_ext=True):
    """
    Load all .npy files in a folder into a dictionary.

    :param folder_path: Path to folder containing .npy files
    :param remove_ext: Whether to remove '.npy' from the keys
    :return: dict {filename(str): numpy array}
    """
    npy_dict = {}
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.npy'):
            key = fname[:-4] if remove_ext else fname  # remove ".npy"
            full_path = os.path.join(folder_path, fname)
            npy_dict[key] = np.load(full_path)
    return npy_dict

def show_data_matrix(data, cmap='viridis'):
    """
    Show 2D matrix as an image using matshow.
    
    :param data: 2D numpy array (e.g., shape: 30 x 180)
    :param cmap: colormap to use (e.g., 'viridis', 'plasma', 'magma', 'gray')
    """
    plt.figure(figsize=(8, 6))
    plt.matshow(data, cmap=cmap, fignum=1)
    plt.colorbar()
    plt.title("Matrix Visualization (matshow)")
    plt.xlabel("Features")
    plt.ylabel("Samples")
    # plt.tight_layout()
    plt.show()

def save_model(model, path, extra_info=None):
    """
    Save AutoEncoder model with architecture info and optional metadata.

    :param model: AutoEncoderClass instance
    :param path: full path to save file (e.g., 'models/ae_model.pt')
    :param extra_info: optional dict for experiment settings (e.g., loss_type, actv name)
    """
    save_dict = {
        "state_dict": model.state_dict(),
        "x_dim": model.input_shape,
        "z_dim": model.z_dim,
        "h_dims": model.h_dims,
        "loss_type": model.loss_type,
        "actv": model.actv.__class__.__name__,  # store only the name of the activation function
    }

    if extra_info is not None:
        save_dict["extra_info"] = extra_info

    os.makedirs(os.path.dirname(path), exist_ok=True)
    th.save(save_dict, path)
    print(f"Model saved to {path}")

def load_saved_model(model_name, device='cpu'):
    """
    Load an AutoEncoderClass instance from the saved config.json and model.pt.
    
    Args:
        model_name (str): folder name under ../../result/
        device (str): 'cpu', 'cuda:0', etc.
    
    Returns:
        ae (AutoEncoderClass): restored model instance
        config (dict): configuration
    """
    save_dir = os.path.join("../../result", model_name)
    config_path = os.path.join(save_dir, "config.json")
    model_path = os.path.join(save_dir, "model.pt")

    # load config
    with open(config_path, 'r') as f:
        config = json.load(f)

    # initialize the model
    actv_map = {
        'Tanh': th.nn.Tanh(),
        'ReLU': th.nn.ReLU(),
        'SiLU': th.nn.SiLU(),
        'GELU': th.nn.GELU(),
        # add if needed
    }
    actv_fn = actv_map[config['actv']]

    ae = AutoEncoderClass(
        x_dim=config['x_dim'][0],  # should later be set from the data
        z_dim=config['z_dim'],
        h_dims=config['h_dims'],
        loss_type=config['loss_type'],
        actv=actv_fn,
        iso=config['iso'],
        iso_reg=config['iso_reg'],
    ).to(device)

    # load model parameters
    ae.load_state_dict(th.load(model_path, map_location=device))
    ae.eval()

    return ae, config

print ("Ready.")

In [ ]:
# Load data
repr_kind = 'screw'
data_path = '../../data/mjcf_test/'+repr_kind + '/'
loaded_data = load_npy_as_dict(data_path)
for k_idx,key in enumerate(loaded_data.keys()):
    print ("[%d/%d] key:[%s] shape:%s"%
           (k_idx,len(loaded_data.keys()),key,loaded_data[key].shape))

In [ ]:
# data = loaded_data['data'][:,[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],:].reshape((30,-1)) # (30x180)
data = load_data_subset(data_path)
show_data_matrix(data, cmap='viridis')

In [ ]:
def compute_hidden_dims(flattened_dim, z_dim):
    mid1 = int(np.sqrt(flattened_dim * z_dim))
    mid2 = int(np.sqrt(mid1 * z_dim))
    return [mid1, mid2]

# run_experiment lives in package/yoon/ae.py. It never overwrites an existing checkpoint:
# if result/<model_name>/ exists, the new model is saved as <model_name>_<timestamp>.
from ae import run_experiment

# ---- Output location for re-runs ----
# The paper results in ./{easy,middle,hard}_motion_*/ and ./all_*/ are read-only references
# (tutorial 03 reads them). Anything this notebook writes goes to OUT_ROOT instead.
OUT_ROOT = './runs'
def out(rel_path):
    p = os.path.join(OUT_ROOT, rel_path)
    os.makedirs(os.path.dirname(p) or '.', exist_ok=True)
    return p

print ("Ready.")


In [ ]:
device="cuda:0"

In [ ]:
if train:
    for idx_z in range(67,121):
        if idx_z!=2:
            results, model_name = run_experiment(plot=False, 
                data_np=data,
                h_dims=[64,32],
                z_dim=idx_z,
                num_epochs=1000,
                lr=1e-3,
                actv=nn.Tanh(),
                loss_type='l2',
                device=device,
                batch_size=10,
                iso=True,
                iso_reg=1e-7,
                # model_name='dh'
                )
        else:
            pass

In [ ]:
model_name='h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10'
# model_name='dh'

In [ ]:
ae, config = load_saved_model(model_name=model_name, device="cpu")
print(f"Hidden dims: {ae.h_dims}")

In [ ]:
robot_chars = RobotCharacteristics()
# label_data = np.load('../../data/mjcf_test/dh/label.npy')
# njnt_data = np.load('../../data/mjcf_test/dh/njnt.npy')
# indices_data = np.load('../../data/mjcf_test/dh/indices.npy')
label_data = np.load('../../data/mjcf_test/'+repr_kind+'/label.npy')
njnt_data = np.load('../../data/mjcf_test/'+repr_kind+'/njnt.npy')
indices_data = np.load('../../data/mjcf_test/'+repr_kind+'/indices.npy')
robot_chars.max_joints_per_group=njnt_data

In [ ]:
print(njnt_data)

In [ ]:
# robot_chars.explore_latent_dimension(data_np = data, model = ae,)

SMPL

In [ ]:
# pkl_path = '../../data/smpl_model/model.pkl'
# file_path = '../../data/mdm_motion/class3/samples_humanml_enc_512_50steps_000750000_seed10_Dance_the_Samba_with_quick_foot_steps_and_rhythmic_arm_swings/sample00_rep01_smpl_params.npy'
# motion_type = 'smpl'

CMU

In [ ]:
pkl_path = None,
file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
motion_type = 'cmu'

In [ ]:
def sum_without_negatives(disparity):
    # sum only the elements other than -999
    total_sum = sum(value for value in disparity if value != -999)

    return total_sum

def get_smpl_pos(chains,joi_body_name,base_between_pelvis=True):
    arrays=[]
    joi_body_name_key = list(joi_body_name.keys())
    joi_body_name_key = [joi.replace('spine','spine1')for joi in joi_body_name_key]
    for i in range(len(chains)):
        smallest_arrays=[]
        for j in range(len(joi_body_name_key)):
            if joi_body_name_key[j] in chains[i].joint_names:
                if base_between_pelvis and joi_body_name_key[j]=='base':
                    p_rp,_ = chains[i].get_joint_pR(joint_name='rp')
                    p_lp,_ = chains[i].get_joint_pR(joint_name='lp')
                    p = 0.5*(p_rp+p_lp)
                else:
                    p,_ = chains[i].get_joint_pR(joint_name=joi_body_name_key[j])
                smallest_arrays.append(p)
        small_arrays=np.array(smallest_arrays)
        arrays.append(small_arrays[np.newaxis,:,:])
    result=np.vstack(arrays)
    return result

def get_cmu_pos(chains,joi_body_name,base_between_pelvis=True, fix_root = False,):
    pairs = {
    'base' : 'Hips',
    'spine' : 'Spine',
    'rs' : 'RightArm',
    're' : 'RightForeArm',
    'rw' : 'RightHand',
    'rh' : 'RightHand',
    'ls' : 'LeftArm',
    'le' : 'LeftForeArm',
    'lw' : 'LeftHand',
    'lh' : 'LeftHand',
    'neck' : 'Neck',
    'head' : 'Head',
    'rp' : 'RightUpLeg',
    'rk' : 'RightLeg',
    'ra' : 'RightFoot',
    'rf' : 'RightFoot',
    'lp' : 'LeftUpLeg',
    'lk' : 'LeftLeg',
    'la' : 'LeftFoot',
    'lf' : 'LeftFoot',
    'rtoe' : 'RightToeBase',
    'ltoe' : 'LeftToeBase',
             }
    arrays=[]
    
    joi_body_name_key = list(joi_body_name.keys())
    for i in range(len(chains)):
        smallest_arrays=[]
        chain = chains[i]
        if fix_root:
            p_root,R_root = chain.get_root_joint_pR()
            chain_origin = deepcopy(chain)
            chain_origin.set_root_joint_pR(p=np.array([0,0,0],R=R_root))
            chain_origin.forward_kinematics()
    
        for j in range(len(joi_body_name_key)):
            
            if joi_body_name_key[j]=='root':
                p_rp,_ = chains[i].get_joint_pR(joint_name=pairs['rp'])
                p_lp,_ = chains[i].get_joint_pR(joint_name=pairs['lp'])
                p = 0.5*(p_rp+p_lp)
                continue
            elif pairs[joi_body_name_key[j]] in chains[i].joint_names:
                if base_between_pelvis and joi_body_name_key[j]=='base':
                    p_rp,_ = chains[i].get_joint_pR(joint_name=pairs['rp'])
                    p_lp,_ = chains[i].get_joint_pR(joint_name=pairs['lp'])
                    p = 0.5*(p_rp+p_lp)
                elif joi_body_name_key[j]=='neck':
                    p_neck,_ = chains[i].get_joint_pR(joint_name=pairs[joi_body_name_key[j]])
                    p_rs,_ = chains[i].get_joint_pR(joint_name=pairs['rs'])
                    p_ls,_ = chains[i].get_joint_pR(joint_name=pairs['ls'])
                    p_neck_z = 0.5*(p_rs+p_ls) 
                    p = np.array([p_neck[0],p_neck[1],p_neck_z[2]])
                else:
                    p,_ = chains[i].get_joint_pR(joint_name=pairs[joi_body_name_key[j]])
            if fix_root:
                p_base,_=chains[i].get_joint_pR(joint_name='Hips')
                p-=p_base
                smallest_arrays.append(p)
                
            else:
                smallest_arrays.append(p)
            # smallest_arrays.append(p)        
        small_arrays=np.array(smallest_arrays)
        # print(f"small arrays: {small_arrays.shape}")
        arrays.append(small_arrays[np.newaxis,:,:])
    result=np.vstack(arrays)
    return result

def r_sq(x1,x2,x_range=1.0,invlen=5.0):
    """
    Scaled pairwise dists 
    """
    x1_scaled,x2_scaled = invlen*x1/x_range,invlen*x2/x_range
    D_sq = distance.cdist(x1_scaled,x2_scaled,'sqeuclidean') 
    return D_sq

def retarget_and_procrustes(
        env,
        pkl_path,
        file_path,
        motion_type = 'smpl', # 'cmu', 'smpl'
        ik_p_spine=False,
        ik_p_neck=False,
        leg_fixed=False,
        ik_R_hand=False,

        animate_chains = False,
        animate_ik=False,
        verbose=False,
        override = None
                  ):
        # Load SMPL model

    joi_body_name=get_joi_body_name_of_irae(override=override)
    joi_smpl_name = list(joi_body_name.keys()) #smpl joi name
    joi_robot_name = list(joi_body_name.values()) # robot body name of joi
    if motion_type=='smpl':
        smpl = SMPLModelClass(model_path=pkl_path)
        # Given file path
    
        # Extracting the base name of the file
        file_name = os.path.basename(file_path)

        # Extract the motion_name
        # Since file_name does not include '/', directly use split on file_path before extracting the file name
        motion_name = file_path.split('seed10_')[1].split('/')[0] if 'seed10_' in file_path else None

        # Extract the sample
        sample = file_name.split('sample00_')[1].split('_smpl')[0] if 'sample00_' in file_name else None

        # Extract the class_name from file_path
        class_name = file_path.split('mdm_motion/')[1].split('/samples_')[0] if 'mdm_motion/' in file_path and '/samples_' in file_path else None
        if verbose:
            print(f"motion_name: {motion_name}")
            print(f"sample: {sample}")
            print(f"class_name: {class_name}")
        
        secs, chains = get_chains_from_smpl(
            file_path        = file_path,
            smpl             = smpl,
            intp_traj        = False,
            plot_chain_graph = False)
        # print(f"motion_type: {motion_type}\n chains: {chains}")
        motion_pos=get_smpl_pos(chains,joi_body_name)
    elif motion_type=='cmu':
        secs, chains = get_chains_from_bvh_cmu_mocap(
            bvh_path = file_path,
            plot_chain_graph = False,
            verbose = False,
        )
        # print(f"motion_type: {motion_type}\n chains: {chains}")
        motion_pos=get_cmu_pos(chains,joi_body_name)
    
    
    L = len(chains); dt = secs[1]-secs[0]; HZ = int(1/dt)
    

    if animate_chains:
        animate_chains_slider(env,secs,chains)
    
    if verbose:
        print ("[%s] loaded. L:[%d] dt:[%.4f]sec HZ:[%d]"%(file_path,L,dt,HZ))

    # preretarget
    qpos_dir_list = directional_preretarget_yoon(
    env,
    secs,
    chains,
    ik_p_spine=ik_p_spine,
    ik_p_neck=ik_p_neck,
    leg_fixed=leg_fixed,
    src_name           = motion_type,
    trgt_name          = 'irae',
    joint_names_for_ik = None,
    max_ik_tick        = 100, # maximum IK iteration
    ik_stepsize        = 1, # IK stepsize
    ik_err_th          = 1e-3, # IK error threshold
    print_ik_failure   = False,
    ik_R_hand          = ik_R_hand,
    ik_R_foot          = not leg_fixed,
    reset_env          = True,
    animate            = animate_ik, # animate pre-retarget process
    plot_joi_src       = False,
    plot_joi_trgt      = True,
    render_every       = 1,
    verbose            = verbose,
    print_every        = 30,
    pause_each_tick    = False,
    post_rigging       = False,
    override           = override,
    )
    # Initialize MuJoCo viewer
    env.reset(step=True)
    if animate_ik:
        env.init_viewer(
            title='MDM to auto_gen Pre-Retarget',
            transparent=True)

    motion_length = L
    num_bodies = env.n_body

    robot_joi_pos = np.zeros((motion_length,len(joi_robot_name),3))
    for tick in range(motion_length):
        env.forward(q=qpos_dir_list[tick]) # FK
        for body_idx in range(num_bodies):
            if body_idx == 0:
                continue
            body_name = env.body_names[body_idx]
            
            if body_name in joi_robot_name:
                # Get the current position of the body
                joi_smpl_idx = [i for i, value in enumerate(joi_robot_name) if value == body_name]
                for i in range(len(joi_smpl_idx)):
                    body_pos = env.data.body(body_name).xpos
                    robot_joi_pos[tick,joi_smpl_idx[i],:]=body_pos

    """ SMPL JOIs have no root! + Procrustes analysis must be run per body
        evaluate smpl_pos(motion_length,11,3) against robot_joi_pos(motion_length,12,3) per JOI
    """

    # drop the root (:,0,:) of robot_joi_pos to match smpl_pos, giving shape (120,11,3)
    robot_joi_pos_wo_root = np.delete(robot_joi_pos, 0, axis=1) # drop element 0 along axis 1
    _ = joi_smpl_name.pop(0)
    procrustes_results = []

    # # run Procrustes analysis

    """Assuming this is done per joint"""
    for i in range(motion_pos.shape[1]):
        matrix1 = motion_pos[:, i, :]
        matrix2 = robot_joi_pos_wo_root[:, i, :]
        
        try:
            # run Procrustes analysis for each i
            mtx1, mtx2, disparity = procrustes(matrix1, matrix2)
            procrustes_results.append((mtx1, mtx2, disparity))
        except:
            procrustes_results.append((matrix1, matrix2, -999))
        
    result = []
    # print example results
    for i, (mtx1, mtx2, disparity) in enumerate(procrustes_results):
        result.append(disparity)
        if verbose:
            print(f"Procrustes analysis for column {i} - joi name: {joi_smpl_name[i]} - Disparity: {disparity:.3f}")
            print("disparity: ", result)

            print("Done.")
    if animate_ik:
        env.viewer.close()
    return sum_without_negatives(result)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import distance

np.random.seed(seed=1)
def override_joi(env,repr_kind):
    body_names = env.body_names
    
    # keep the names to match in each group
    group_map = {
        'base_link': [],
        'torso': [],
        'neck': [],
        'right_shoulder_girdle': [],
        'right_shoulder': [],
        'right_upperarm': [],
        'right_elbow': [],
        'right_forearm': [],
        'right_wrist': [],
        'left_shoulder_girdle': [],
        'left_shoulder': [],
        'left_upperarm': [],
        'left_elbow': [],
        'left_forearm': [],
        'left_wrist': [],
    }

    # classify names
    for name in body_names:
        for group in group_map.keys():
            if group in name:
                group_map[group].append(name)

    # set the fallback path
    fallback_order_screw = {
        'right_elbow': ['right_upperarm', 'right_shoulder', 'right_shoulder_girdle', 'torso', 'base_link'],
        'right_wrist': ['right_forearm', 'right_elbow', 'right_upperarm', 'right_shoulder', 'right_shoulder_girdle', 'torso', 'base_link'],
        'left_elbow': ['left_upperarm', 'left_shoulder', 'left_shoulder_girdle', 'torso', 'base_link'],
        'left_wrist': ['left_forearm', 'left_elbow', 'left_upperarm', 'left_shoulder', 'left_shoulder_girdle', 'torso', 'base_link'],
        'torso': ['base_link'],
        'neck': ['torso', 'base_link'],
        'right_shoulder': ['right_shoulder_girdle', 'torso', 'base_link'],
        'left_shoulder': ['left_shoulder_girdle', 'torso', 'base_link'],
    }
    fallback_order_dh = {
        'right_elbow': ['right_upperarm', 'right_shoulder', 'right_shoulder_girdle',  'torso', 'base_link'],
        'right_tcp': ['right_wrist','right_forearm', 'right_elbow', 'right_upperarm', 'right_shoulder', 'right_shoulder_girdle',  'torso', 'base_link'],
        'left_elbow': ['left_upperarm', 'left_shoulder', 'left_shoulder_girdle',  'torso', 'base_link'],
        'left_tcp': ['left_wrist','left_forearm', 'left_elbow', 'left_upperarm', 'left_shoulder', 'left_shoulder_girdle',  'torso', 'base_link'],
        'torso': ['base_link'],
        'neck': ['torso', 'base_link'],
        'right_shoulder': ['right_shoulder_girdle',  'torso', 'base_link'],
        'left_shoulder': ['left_shoulder_girdle',  'torso', 'base_link'],
    }
    if repr_kind=="screw":
        fallback_order=fallback_order_screw
    elif repr_kind=="dh":
        fallback_order=fallback_order_dh
    targets = ['torso', 'neck', 'right_shoulder', 'right_elbow', 'right_wrist',
               'left_shoulder', 'left_elbow', 'left_wrist']
    result_name    = ['spine', 'neck', 'rs', 're', 'rw', 'ls', 'le', 'lw']

    results = {}

    for idx, group in enumerate(targets):
        # find directly
        matches = group_map.get(group, [])
        joi_type = result_name[idx]
        if matches:
            selected = matches[int(len(matches) / 2)]
            results[joi_type] = selected
            if joi_type =='rw':
                results['rh'] = selected
            if joi_type =='lw':
                results['lh'] = selected
            continue

        # take the fallback
        fallback_list = fallback_order.get(group, [])
        found = False
        for fallback in fallback_list:
            candidates = group_map.get(fallback, [])
            if candidates:
                selected = candidates[-1]  # last one
                results[joi_type] = selected
                found = True
                if joi_type =='rw':
                    results['rh'] = selected
                if joi_type =='lw':
                    results['lh'] = selected
                break
        if not found:
            continue

    return results

def evaluate(x,
             model,
             robot_chars, 
             motion_type,
             pkl_path,
             file_path,
             repr_kind="screw"):
    results= np.zeros((x.shape[0],1))
    decode = model.decode(th.tensor(x,device=next(model.parameters()).device,dtype=th.float32)).detach().numpy()
    full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode,threshold=0.5,repr_kind=repr_kind)
    data_to_visualize = robot_chars.convert_flat_interp_to_dict(flat_interp_data = full_decode,repr_kind=repr_kind)
    # print(f"data_to_vis: {data_to_visualize}")
    robot_chars.visualize_data_to_classify(data_to_visualize, save_video=False,vis=False, xml=True, repr_kind = repr_kind)
    i=0
    for name, xml in robot_chars.xml.items():
        
        env = MuJoCoParserClass(name=name, xml_string=xml, verbose=False)
        if len(env.body_names)<=2:
            results[i]=10.0
            i+=1
            continue
        things_to_override = override_joi(env,repr_kind)
        # print(env.body_names)
        try:
            result = retarget_and_procrustes(
                        env,
                        pkl_path=pkl_path,
                        file_path=file_path,
                        motion_type = motion_type,
                        ik_p_spine=False,
                        ik_p_neck=True,
                        leg_fixed=True,
                        ik_R_hand=False,
                        animate_chains = False,
                        animate_ik=False,
                        verbose=False,
                        override= things_to_override,
                        )
        except Exception as e:
            print(f"Error during evaluation for model {x[i]}: {e}\n body_names: {env.body_names}")
            result = 10.0
        if i%500==499: print(f"i: {i} result: {result}")
        results[i]=result
        i+=1
    return results

def r_sq(x1, x2, x_range=1.0, invlen=5.0):
    x1_scaled, x2_scaled = invlen * x1 / x_range, invlen * x2 / x_range
    return distance.cdist(x1_scaled, x2_scaled, 'sqeuclidean')

# Range
x_dim = ae.z_dim
x_minval, x_maxval = -10,10

# collect initial data
x_voo = x_minval + (x_maxval - x_minval) * np.random.rand(5, x_dim)
# print(f"x_voo: {x_voo}")
y_voo = evaluate(x_voo,ae, robot_chars, motion_type,pkl_path,file_path,repr_kind)
# print(f"y_voo: {y_voo}")

max_iter = 10
for it in range(max_iter):
    # print(f"VOO iter: {it}")
    # VOO
    if np.random.rand() < 0.5:
        x_sel = x_minval + (x_maxval - x_minval) * np.random.rand(1, x_dim)
    else:
        idx_min = np.argmin(y_voo)
        x_min_pt = x_voo[idx_min, :].reshape((1, -1))
        n = x_voo.shape[0] - 1
        dists = r_sq(x_min_pt, np.delete(x_voo, [idx_min], axis=0), invlen=1.0)
        dist_min = np.min(dists)
        n_loop, max_loop = 0, int(1e3)
        while True:
            n_loop += 1
            if n_loop < 10:
                x_sel = x_minval + (x_maxval - x_minval) * np.random.rand(1, x_dim)
            else:
                x_sel = x_min_pt + np.random.randn(1, x_dim)
            dist_sel = r_sq(x_min_pt, x_sel)
            if dist_sel < dist_min:
                break

    # evaluate and append the data
    y_sel = evaluate(x_sel,model=ae,robot_chars=robot_chars,motion_type=motion_type,pkl_path=pkl_path,file_path=file_path,repr_kind=repr_kind)
    x_voo = np.concatenate((x_voo, x_sel), axis=0)
    y_voo = np.concatenate((y_voo, y_sel), axis=0)
print("Done.")

In [ ]:
# # Meshgrid for function surface
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy

# x1 = np.linspace(x_minval, x_maxval, 50)
# x2 = np.linspace(x_minval, x_maxval, 50)
# X1, X2 = np.meshgrid(x1, x2)
# X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
# print(f"X_grid: {X_grid}")
# Z = evaluate(X_grid,ae,robot_chars,motion_type,pkl_path,file_path,repr_kind)

# Z1 = Z.reshape(X1.shape)
# np.save(out('easy_motion_screw/X1.npy'),X1)
# np.save(out('easy_motion_screw/X2.npy'),X2)
# np.save(out('easy_motion_screw/Z1.npy'),Z1)
print("Starting meshgrid")
file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle

x1 = np.linspace(x_minval, x_maxval, 50)
x2 = np.linspace(x_minval, x_maxval, 50)
X1, X2 = np.meshgrid(x1, x2)
X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
print(f"X_grid: {X_grid}")
Z = evaluate(X_grid,ae,robot_chars,motion_type,pkl_path,file_path,repr_kind)

Z1 = Z.reshape(X1.shape)
np.save(out('middle_motion_screw/X1.npy'),X1)
np.save(out('middle_motion_screw/X2.npy'),X2)
np.save(out('middle_motion_screw/Z1.npy'),Z1)

file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
x1 = np.linspace(x_minval, x_maxval, 50)
x2 = np.linspace(x_minval, x_maxval, 50)
X1, X2 = np.meshgrid(x1, x2)
X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
print(f"X_grid: {X_grid}")
Z = evaluate(X_grid,ae,robot_chars,motion_type,pkl_path,file_path,repr_kind)

Z1 = Z.reshape(X1.shape)
np.save(out('hard_motion_screw/X1.npy'),X1)
np.save(out('hard_motion_screw/X2.npy'),X2)
np.save(out('hard_motion_screw/Z1.npy'),Z1)

# ---- Plot with Meshgrid ---- #
# X1 = np.load('./easy_motion_screw/X1.npy')
# X2 = np.load('./easy_motion_screw/X2.npy')
# Z1 = np.load('./easy_motion_screw/Z1.npy')
# fig = plt.figure(figsize=(12, 8))
# ax = fig.add_subplot(111, projection='3d')
# # Surface
# ax.plot_surface(X1, X2, Z1, cmap='viridis', alpha=0.4, edgecolor='none')

# # Data points
# ax.scatter(x_voo[:,0], x_voo[:,1], y_voo[:,0], c='blue', s=50, label='voo_explored')
# ax.scatter(x_sel[0,0], x_sel[0,1], y_sel[0,0], c='red', s=500, label='voo_sel', edgecolors='k')
# print(f"optimal: {x_sel}, procrustes: {y_sel}")
# ax.set_xlabel('x1')
# ax.set_ylabel('x2')
# ax.set_zlabel('Evaluation')
# ax.set_title(f"VOO", fontsize=14)
# ax.legend()
# plt.tight_layout()

# plt.savefig('./easy_motion_screw/meshgrid_result_easy.png')
# plt.show()

In [ ]:
# pkl_path = None,
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
# motion_type = 'cmu'
# # Meshgrid for function surface
# x1 = np.linspace(x_minval, x_maxval, 50)
# x2 = np.linspace(x_minval, x_maxval, 50)
# X1, X2 = np.meshgrid(x1, x2)
# X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
# # print(f"X_grid: {X_grid}")
# Z = evaluate(X_grid,model=ae,robot_chars=robot_chars,motion_type=motion_type,pkl_path=pkl_path,file_path=file_path,repr_kind=repr_kind)

# Z1 = Z.reshape(X1.shape)
# np.save(out('easy_motion_dh/X1.npy'),X1)
# np.save(out('easy_motion_dh/X2.npy'),X2)
# np.save(out('easy_motion_dh/Z1.npy'),Z1)


# print("Done")

In [ ]:
# pkl_path = None,
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
# motion_type = 'cmu'
# # Meshgrid for function surface
# x1 = np.linspace(x_minval, x_maxval, 50)
# x2 = np.linspace(x_minval, x_maxval, 50)
# X1, X2 = np.meshgrid(x1, x2)
# X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
# # print(f"X_grid: {X_grid}")
# Z = evaluate(X_grid,model=ae,robot_chars=robot_chars,motion_type=motion_type,pkl_path=pkl_path,file_path=file_path,repr_kind=repr_kind)

# Z1 = Z.reshape(X1.shape)
# np.save(out('middle_motion_dh/X1.npy'),X1)
# np.save(out('middle_motion_dh/X2.npy'),X2)
# np.save(out('middle_motion_dh/Z1.npy'),Z1)


In [ ]:
# pkl_path = None,
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
# # file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
# motion_type = 'cmu'
# # Meshgrid for function surface
# x1 = np.linspace(x_minval, x_maxval, 50)
# x2 = np.linspace(x_minval, x_maxval, 50)
# X1, X2 = np.meshgrid(x1, x2)
# X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1)
# # print(f"X_grid: {X_grid}")
# Z = evaluate(X_grid,model=ae,robot_chars=robot_chars,motion_type=motion_type,pkl_path=pkl_path,file_path=file_path,repr_kind=repr_kind)

# Z1 = Z.reshape(X1.shape)
# np.save(out('hard_motion_dh/X1.npy'),X1)
# np.save(out('hard_motion_dh/X2.npy'),X2)
# np.save(out('hard_motion_dh/Z1.npy'),Z1)

### Visualize results

In [ ]:
# motion_level = 'hard'
# x_1 = np.load(f'./{motion_level}_motion/X1.npy')
# x_2 = np.load(f'./{motion_level}_motion/X2.npy')
# z_1 = np.load(f'./{motion_level}_motion/Z1.npy')

# # ---- Plot with Meshgrid ---- #
# fig = plt.figure(figsize=(12, 8))
# ax = fig.add_subplot(111, projection='3d')
# # Surface
# # ax.plot_surface(x_1, x_2, z_1, cmap='viridis', alpha=0.4, edgecolor='none')
# # ax.contourf(x_1,x_2,z_1,levels=10,cmap='viridis')
# ax.contourf(x_1,x_2,z_1, zdir='z', offset=1, cmap='viridis')
# # Data points
# ax.set_xlabel('x1')
# ax.set_ylabel('x2')
# ax.set_zlabel('Evaluation')
# ax.set_title(f"VOO", fontsize=14)
# ax.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# fig = plt.figure(figsize=(12, 8))
# ax = fig.add_subplot(111)

# # 2D Contour plot
# contour = ax.contourf(x_1, x_2, z_1, levels=10, cmap='viridis')

# # Data points (add here if any)
# # ax.scatter(...)

# # Labels and title
# ax.set_xlabel('x1')
# ax.set_ylabel('x2')
# ax.set_title("VOO", fontsize=14)

# # Colorbar
# fig.colorbar(contour, ax=ax, label='Evaluation')

# plt.tight_layout()
# plt.show()

In [ ]:
# easy_optimal = np.array([[5.39344407, 7.67325486]])
# middle_optimal = np.array([[-7.120334, -8.1370581]])
# hard_optimal = np.array([[8.13185319, -7.43121962]])


In [ ]:
# decode = ae.decode(th.tensor(easy_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
# full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode,threshold=0.5)
# data_to_visualize = robot_chars.convert_flat_interp_to_dict(full_decode)

# clip = robot_chars.visualize_data_to_classify(data_to_visualize, save_video=True,vis=True, test = True, xml=True)
# clip.ipython_display()

In [ ]:
# decode2 = ae.decode(th.tensor(middle_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
# full_decode2=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode2,threshold=0.5)
# data_to_visualize2 = robot_chars.convert_flat_interp_to_dict(full_decode2)

# clip2 = robot_chars.visualize_data_to_classify(data_to_visualize2, save_video=True,vis=True, test=True, xml=True)
# clip2.ipython_display()

In [ ]:
# decode3 = ae.decode(th.tensor(hard_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
# full_decode3=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode3,threshold=0.5)
# data_to_visualize3 = robot_chars.convert_flat_interp_to_dict(full_decode3)

# clip3 = robot_chars.visualize_data_to_classify(data_to_visualize3, save_video=True,vis=True,test=True, xml=True)
# clip3.ipython_display()